# Reproducibility and Agreement of LLM-Generated Rankings

This notebook reads the archived LLM responses and reproduces the ranking analyses used
in the manuscript. It makes no API calls and requires no patient-level data.

Primary outputs are the run-level rank-variability table, the Kendall agreement matrix,
the supplementary feature-ranking table, and manuscript Figure 3. The removed exploratory
analyses (Spearman correlation, RBO, variance decomposition, and rank-distribution plots)
are intentionally not part of this submission workflow.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root from either the root or notebooks directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "notebooks").is_dir() and (candidate / ".gitignore").is_file():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


PROJECT_ROOT = find_project_root()

import itertools
import json

import numpy as np
import pandas as pd
from scipy.stats import kendalltau

POOL_FILE = PROJECT_ROOT / "data" / "derived" / "feature_pool.txt"
RAW_DIR = PROJECT_ROOT / "results" / "llm_runs"
AGG_DIR = PROJECT_ROOT / "results" / "aggregated"
OUT = PROJECT_ROOT / "results" / "ranking"
FIGURE_DIR = PROJECT_ROOT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RUNS = [1, 2, 3, 4, 5]
MODELS = ["gpt", "gemini", "claude"]
METHODS = ["score", "rank", "seq"]
MODEL_LABEL = {
    "gpt": "GPT-4.1",
    "gemini": "Gemini 2.5 Pro",
    "claude": "Claude Sonnet 4.5",
}
METHOD_LABEL = {
    "score": "Scoring",
    "rank": "Ranking",
    "seq": "Sequencing",
    "lmprior": "Log-prior scoring",
}
VALUE_FIELD = {"score": "score", "rank": "rank", "seq": "step", "lmprior": "score"}
HIGHER_IS_BETTER = {"score", "lmprior"}

POOL = [line.strip() for line in POOL_FILE.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
POOL_SET = set(POOL)
if len(POOL) != 107 or len(POOL_SET) != 107:
    raise ValueError("The feature pool must contain 107 unique features.")
CANONICAL = {feature.lower(): feature for feature in POOL}
print(f"Loaded {len(POOL)} candidate features and archived outputs from {RAW_DIR}")


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        if line.strip():
            rows.append(json.loads(line))
    return rows


def raw_path(method: str, model: str, run: int) -> Path:
    folder = "log_prior_scoring" if method == "lmprior" else method
    return RAW_DIR / folder / model / f"run_{run:02d}.jsonl"


def canonical(name: str) -> str:
    return CANONICAL.get(str(name).strip().lower(), str(name).strip())


def per_run_long(method: str, model: str) -> pd.DataFrame:
    rows = []
    field = VALUE_FIELD[method]
    for run in RUNS:
        frame = pd.DataFrame(read_jsonl(raw_path(method, model, run)))
        frame["feature"] = frame["feature"].map(canonical)
        values = frame[field].astype(float)
        if method in HIGHER_IS_BETTER:
            values = values.round(6).rank(method="min", ascending=False)
        for feature, rank in zip(frame["feature"], values):
            rows.append({"feature": feature, "run": run, "rank": float(rank)})
    return pd.DataFrame(rows)


def aggregated_vector(method: str, model: str) -> pd.DataFrame:
    if method == "score":
        frames = []
        for run in RUNS:
            frame = pd.DataFrame(read_jsonl(raw_path(method, model, run)))[["feature", "score"]]
            frame["feature"] = frame["feature"].map(canonical)
            frames.append(frame.rename(columns={"score": f"score_{run}"}))
        merged = frames[0]
        for frame in frames[1:]:
            merged = merged.merge(frame, on="feature", validate="one_to_one")
        score_columns = [f"score_{run}" for run in RUNS]
        merged["mean_score"] = merged[score_columns].astype(float).mean(axis=1).round(6)
        merged["rank"] = merged["mean_score"].rank(method="min", ascending=False)
        return merged[["feature", "rank"]].sort_values("rank").reset_index(drop=True)

    long = per_run_long(method, model)
    vector = long.groupby("feature", as_index=False)["rank"].mean()
    return vector.sort_values("rank").reset_index(drop=True)


expected = [(method, model) for method in METHODS for model in MODELS] + [("lmprior", "gpt")]
inventory = []
for method, model in expected:
    for run in RUNS:
        path = raw_path(method, model, run)
        rows = read_jsonl(path)
        features = {canonical(row["feature"]) for row in rows}
        status = "OK" if len(rows) == 107 and features == POOL_SET else "CHECK"
        inventory.append({"procedure": METHOD_LABEL[method], "model": MODEL_LABEL[model],
                          "run": run, "records": len(rows), "status": status})

inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT / "run_inventory.csv", index=False, encoding="utf-8-sig")
if not inventory["status"].eq("OK").all():
    raise ValueError("At least one archived run failed the integrity check.")
print(f"Validated {len(inventory)} run files ({inventory.records.sum():,} feature records).")


In [ ]:
VECTORS = {
    (method, model): aggregated_vector(method, model)
    for method in METHODS
    for model in MODELS
}
LOG_PRIOR_VECTOR = aggregated_vector("lmprior", "gpt")

# Verify the nine primary model-procedure columns against the frozen aggregate table.
# The archived GPT-4.1 Scoring table contains one floating-point tie split outside the
# top-30 budget (0.688 versus 0.688). The manuscript analysis rounds mean scores to six
# decimals before assigning minimum ranks, so this check allows tie-only rank relabelling
# while requiring the selected top-30 set to remain identical.
frozen = pd.read_csv(AGG_DIR / "combined_ranks.csv", encoding="utf-8-sig")
for method in METHODS:
    for model in MODELS:
        column = f"{method}_{model}"
        observed = VECTORS[(method, model)].rename(columns={"rank": "observed"})
        expected_values = frozen[["feature", column]].rename(columns={column: "expected"})
        check = observed.merge(expected_values, on="feature", validate="one_to_one")
        observed_top30 = set(check.nsmallest(30, "observed")["feature"])
        expected_top30 = set(check.nsmallest(30, "expected")["feature"])
        tau = kendalltau(check["observed"], check["expected"]).statistic
        if observed_top30 != expected_top30 or tau < 0.999:
            raise ValueError(f"Aggregated ordering mismatch for {column}.")
        if not np.allclose(check["observed"], check["expected"], atol=1e-8):
            print(f"Validated {column}: equivalent ordering stored on a different rank scale.")
print("Computed rankings are consistent with results/aggregated/combined_ranks.csv.")


## Run-to-run rank variability

For each model-procedure combination, feature-specific rank standard deviations are
calculated across five runs. The table reports the mean among the 30 most variable
features, matching the manuscript definition.

In [ ]:
def mean_top_variable_rank_sd(method: str, model: str, top_n: int = 30) -> float:
    pivot = per_run_long(method, model).pivot(index="feature", columns="run", values="rank")
    rank_sd = pivot.std(axis=1, ddof=1)
    rank_sd = rank_sd.mask(np.isclose(rank_sd, 0.0, atol=1e-8), 0.0)
    return float(rank_sd.nlargest(top_n).mean())


rows = []
for model in MODELS:
    rows.append({
        "Model": MODEL_LABEL[model],
        **{METHOD_LABEL[method]: round(mean_top_variable_rank_sd(method, model), 4)
           for method in METHODS},
    })
rank_variability = pd.DataFrame(rows)
rank_variability.to_csv(OUT / "Table2_rank_variability.csv", index=False, encoding="utf-8-sig")

log_prior_rank_sd = mean_top_variable_rank_sd("lmprior", "gpt")
print(rank_variability.to_string(index=False))
print(f"Log-prior scoring / GPT-4.1: {log_prior_rank_sd:.4f}")


## Kendall rank agreement

Diagonal cells are the mean Kendall tau-b across the ten run pairs from the same
model-procedure combination. Off-diagonal cells compare five-run aggregated rankings.

In [ ]:
PROCEDURES = [(method, model) for method in METHODS for model in MODELS]
LABELS = [f"{METHOD_LABEL[method]} / {MODEL_LABEL[model]}" for method, model in PROCEDURES]

agreement = np.full((len(PROCEDURES), len(PROCEDURES)), np.nan)
for i, (method_i, model_i) in enumerate(PROCEDURES):
    run_matrix = per_run_long(method_i, model_i).pivot(index="feature", columns="run", values="rank")
    agreement[i, i] = np.mean([
        kendalltau(run_matrix[a], run_matrix[b]).statistic
        for a, b in itertools.combinations(RUNS, 2)
    ])
    for j, (method_j, model_j) in enumerate(PROCEDURES):
        if i == j:
            continue
        merged = VECTORS[(method_i, model_i)].merge(
            VECTORS[(method_j, model_j)], on="feature", suffixes=("_i", "_j")
        )
        agreement[i, j] = kendalltau(merged["rank_i"], merged["rank_j"]).statistic

agreement_frame = pd.DataFrame(agreement, index=LABELS, columns=LABELS)
agreement_frame.to_csv(OUT / "kendall_agreement_matrix.csv", encoding="utf-8-sig")

pair_rows = []
for i in range(len(PROCEDURES)):
    for j in range(i + 1, len(PROCEDURES)):
        method_i, model_i = PROCEDURES[i]
        method_j, model_j = PROCEDURES[j]
        if method_i == method_j:
            comparison = "same procedure, different model"
        elif model_i == model_j:
            comparison = "same model, different procedure"
        else:
            comparison = "different model and procedure"
        pair_rows.append({"procedure_1": LABELS[i], "procedure_2": LABELS[j],
                          "comparison": comparison, "kendall_tau_b": agreement[i, j]})
pd.DataFrame(pair_rows).to_csv(OUT / "kendall_pairwise_agreement.csv",
                               index=False, encoding="utf-8-sig")
print(f"Diagonal agreement range: {np.diag(agreement).min():.3f}-{np.diag(agreement).max():.3f}")


## Score and log-prior diagnostics

These diagnostics support the secondary description of score resolution and verify that
both Y and N tokens were observed for every Log-prior scoring call.

In [ ]:
resolution_rows = []
for model in MODELS:
    raw = pd.concat([
        pd.DataFrame(read_jsonl(raw_path("score", model, run))).assign(run=run)
        for run in RUNS
    ], ignore_index=True)
    raw["score"] = raw["score"].astype(float).round(6)
    mean_score = raw.groupby("feature")["score"].mean().round(6).sort_values(ascending=False)
    cutoff = float(mean_score.iloc[29])
    n_above = int((mean_score > cutoff).sum())
    n_tied = int((mean_score == cutoff).sum())
    resolution_rows.append({
        "model": MODEL_LABEL[model],
        "observations": len(raw),
        "distinct_run_level_scores": raw["score"].nunique(),
        "top30_cutoff_score": cutoff,
        "features_above_cutoff": n_above,
        "features_tied_at_cutoff": n_tied,
        "tie_crosses_top30_cutoff": n_above < 30 < n_above + n_tied,
    })
score_resolution = pd.DataFrame(resolution_rows)
score_resolution.to_csv(OUT / "score_resolution.csv", index=False, encoding="utf-8-sig")

log_prior = pd.concat([
    pd.DataFrame(read_jsonl(raw_path("lmprior", "gpt", run))).assign(run=run)
    for run in RUNS
], ignore_index=True)
log_prior_diagnostics = pd.DataFrame([{
    "observations": len(log_prior),
    "distinct_scores": log_prior["score"].astype(float).nunique(),
    "Y_not_observed": int((~log_prior["y_observed"].astype(bool)).sum()),
    "N_not_observed": int((~log_prior["n_observed"].astype(bool)).sum()),
    "mean_top_variable_rank_sd": round(log_prior_rank_sd, 4),
}])
log_prior_diagnostics.to_csv(OUT / "log_prior_diagnostics.csv",
                             index=False, encoding="utf-8-sig")
print(score_resolution.to_string(index=False))
print(log_prior_diagnostics.to_string(index=False))


## Feature-level rankings

The following cell creates the complete supplementary ranking table and a compact table
of the top 30 features from each procedure.

In [ ]:
all_vectors = dict(VECTORS)
all_vectors[("lmprior", "gpt")] = LOG_PRIOR_VECTOR

rank_table = pd.DataFrame({"Feature": POOL})
for (method, model), vector in all_vectors.items():
    label = f"{METHOD_LABEL[method]} / {MODEL_LABEL[model]}"
    rank_table = rank_table.merge(
        vector.rename(columns={"feature": "Feature", "rank": label}),
        on="Feature", validate="one_to_one"
    )
rank_columns = [column for column in rank_table.columns if column != "Feature"]
rank_table["Overall mean rank"] = rank_table[rank_columns].mean(axis=1)
rank_table = rank_table.sort_values(["Overall mean rank", "Feature"]).reset_index(drop=True)
rank_table.insert(0, "Overall order", np.arange(1, len(rank_table) + 1))
rank_table.to_csv(OUT / "Supplementary_feature_rankings.csv",
                  index=False, encoding="utf-8-sig")

top30 = {}
for (method, model), vector in all_vectors.items():
    label = f"{METHOD_LABEL[method]} / {MODEL_LABEL[model]}"
    top30[label] = vector.nsmallest(30, "rank")["feature"].tolist()
top30 = pd.DataFrame(top30, index=np.arange(1, 31))
top30.index.name = "rank"
top30.to_csv(OUT / "top30_by_procedure.csv", encoding="utf-8-sig")
rank_table.head(10)


## Figure 3. Kendall agreement among model-procedure combinations

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.figure_export import save_submission_figure


def two_line_label(label: str) -> str:
    procedure, model = label.split(" / ", 1)
    return f"{procedure}\n{model}"


plot_labels = [two_line_label(label) for label in LABELS]
pd.DataFrame(agreement, index=LABELS, columns=LABELS).to_excel(
    FIGURE_DIR / 'Figure3_kendall_agreement.xlsx',
    sheet_name='Kendall tau-b',
)
style = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 7.2,
    "ytick.labelsize": 7.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

with plt.rc_context(style):
    fig, ax = plt.subplots(figsize=(8.15, 7.45))
    cell_edges = np.arange(-0.5, 9.5, 1.0)
    image = ax.pcolormesh(
        cell_edges, cell_edges, agreement, cmap="Greys",
        vmin=0.25, vmax=1.00, shading="flat", rasterized=False,
    )
    ax.set_xlim(-0.5, 8.5)
    ax.set_ylim(8.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks(np.arange(9)); ax.set_yticks(np.arange(9))
    ax.set_xticklabels(plot_labels, rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(plot_labels)
    ax.tick_params(axis="both", which="major", length=0, pad=4)
    ax.set_xticks(np.arange(-0.5, 9, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 9, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.55)
    ax.tick_params(which="minor", bottom=False, left=False)
    for boundary in (2.5, 5.5):
        ax.axvline(boundary, color="black", linewidth=1.25)
        ax.axhline(boundary, color="black", linewidth=1.25)
    for row in range(9):
        for col in range(9):
            value = agreement[row, col]
            ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                    color="white" if value >= 0.70 else "black", fontsize=7.8,
                    fontweight="bold" if row == col else "normal")
    for index in range(9):
        ax.add_patch(Rectangle((index - 0.5, index - 0.5), 1, 1, fill=False,
                               edgecolor="black", linewidth=1.9, zorder=5))
        ax.add_patch(Rectangle((index - 0.47, index - 0.47), 0.94, 0.94, fill=False,
                               edgecolor="white", linewidth=0.65, zorder=6))
    ax.set_title("Kendall rank agreement among model-procedure combinations", pad=12)
    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.035)
    colorbar.solids.set_rasterized(False)
    colorbar.set_label(r"Kendall $\tau$-b", fontsize=8.5)
    colorbar.ax.tick_params(labelsize=7.5)
    fig.subplots_adjust(left=0.27, right=0.91, bottom=0.22, top=0.90)
    save_submission_figure(
        fig, FIGURE_DIR / 'Figure3_kendall_agreement',
        raster_kind='line_drawing', column_width='full',
    )
    plt.show()
    plt.close(fig)
